# Project 1 · Shipping-cost model, trained locally

Predict the **shipping cost** (`freight_value`, in Brazilian reais) of an Olist order item,
using only information known when the customer checks out.

This notebook runs entirely on your computer: no Azure needed. It follows the same steps as
`freight-cost-model/src/train.py`, which runs the same logic as an Azure ML job.

| Step | What happens |
|---|---|
| 1–2 | Find and load the dataset |
| 3 | Build features: one row per order item |
| 4 | Explore the target |
| 5 | Split by time: train before June 2018, test June–August 2018 |
| 6 | Train and compare three models |
| 7 | Explain the best model |
| 8 | Save the model |
| 9 | **Try it in a Gradio app** |

**Dataset:** run root `scripts/bootstrap.py` first. This optional local notebook reads the
shared `datasets/olist/Ecommerce Dataset` folder; cloud training uses `jobs/train-job.yml`.

In [ ]:
# Run once to install the packages (restart the kernel afterwards if anything was installed)
%pip install -q pandas scikit-learn matplotlib gradio joblib

## 1. Find the dataset

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

OLIST_FILES = [
    "olist_customers_dataset.csv", "olist_geolocation_dataset.csv", "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv", "olist_order_reviews_dataset.csv", "olist_orders_dataset.csv",
    "olist_products_dataset.csv", "olist_sellers_dataset.csv", "product_category_name_translation.csv",
]

def find_data_dir():
    """Find the single extracted Olist dataset in the cloned repository."""
    candidates = [folder / 'datasets' / 'olist' / 'Ecommerce Dataset'
                  for folder in [Path.cwd(), *Path.cwd().parents]]
    for candidate in candidates:
        if all((candidate / name).exists() for name in OLIST_FILES):
            return candidate
    raise FileNotFoundError(
        "Olist CSV files not found. Run scripts/bootstrap.py and open this notebook "
        "from inside the cloned repository."
    )

DATA_DIR = find_data_dir()
print("Reading the dataset from:", DATA_DIR)

def load(name):
    return pd.read_csv(DATA_DIR / name)

## 2. Load the tables
The Olist data is spread over linked tables, like a small relational database.

In [ ]:
items = load("olist_order_items_dataset.csv")
orders = load("olist_orders_dataset.csv")
customers = load("olist_customers_dataset.csv")
sellers = load("olist_sellers_dataset.csv")
products = load("olist_products_dataset.csv")
geolocation = load("olist_geolocation_dataset.csv")
translation = load("product_category_name_translation.csv")

for name, table in {"items": items, "orders": orders, "customers": customers, "sellers": sellers,
                    "products": products, "geolocation": geolocation}.items():
    print(f"{name:<12} {table.shape[0]:>9,} rows  {table.shape[1]:>2} columns")
items.head()

## 3. Build the features
Start from **order items** (one row per item sold) and join the purchase date, both zip codes and states, the product size and weight, and the English category name. Then turn the raw columns into features.

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance between two points on Earth, in kilometres (works on arrays too)."""
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 6371.0 * 2 * np.arcsin(np.sqrt(a))

# Average coordinates per zip-code prefix (the geolocation table has many rows per prefix)
zip_coords = geolocation.groupby("geolocation_zip_code_prefix")[["geolocation_lat", "geolocation_lng"]].mean()

# Centre of each state (used later by the Gradio app to estimate a distance from two states)
inside_brazil = geolocation["geolocation_lat"].between(-34, 6) & geolocation["geolocation_lng"].between(-74, -34)
state_centres = geolocation[inside_brazil].groupby("geolocation_state")[["geolocation_lat", "geolocation_lng"]].median()
STATES = sorted(state_centres.index)
print(f"{len(zip_coords):,} zip prefixes, {len(STATES)} states")

In [ ]:
df = (
    items.merge(orders[["order_id", "customer_id", "order_purchase_timestamp"]], on="order_id")
    .merge(customers[["customer_id", "customer_zip_code_prefix", "customer_state"]], on="customer_id")
    .merge(sellers[["seller_id", "seller_zip_code_prefix", "seller_state"]], on="seller_id")
    .merge(products, on="product_id", how="left")
    .merge(translation, on="product_category_name", how="left")
)
df = df.join(zip_coords.add_prefix("customer_"), on="customer_zip_code_prefix")
df = df.join(zip_coords.add_prefix("seller_"), on="seller_zip_code_prefix")

# --- Feature engineering ---
df["distance_km"] = haversine_km(df["seller_geolocation_lat"], df["seller_geolocation_lng"],
                                 df["customer_geolocation_lat"], df["customer_geolocation_lng"])
df["volume_cm3"] = df["product_length_cm"] * df["product_height_cm"] * df["product_width_cm"]
df["same_state"] = (df["seller_state"] == df["customer_state"]).astype(int)
df["items_in_order"] = df.groupby("order_id")["order_item_id"].transform("size")       # cart size
df["items_from_seller"] = df.groupby(["order_id", "seller_id"])["order_item_id"].transform("size")
df["purchase_timestamp"] = pd.to_datetime(df["order_purchase_timestamp"])
df["purchase_month"] = df["purchase_timestamp"].dt.month
df["product_category"] = df["product_category_name_english"].fillna(df["product_category_name"]).fillna("unknown")

TARGET = "freight_value"
CATEGORICAL = ["seller_state", "customer_state", "product_category"]
NUMERIC = ["price", "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm",
           "volume_cm3", "distance_km", "same_state", "items_in_order", "items_from_seller", "purchase_month"]
FEATURES = CATEGORICAL + NUMERIC
for col in CATEGORICAL:
    df[col] = df[col].fillna("unknown").astype(str)
df[NUMERIC] = df[NUMERIC].astype("float64")

print(f"{len(df):,} order items, {len(FEATURES)} features")
df[FEATURES + [TARGET]].head()

## 4. Explore the target
Most items cost R$10–25 to ship, but a few cost hundreds: the target is **skewed**. Distance is clearly related to the price.

In [ ]:
print(df[TARGET].describe().round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df[TARGET].clip(upper=df[TARGET].quantile(0.99)).hist(bins=60, ax=axes[0])
axes[0].set(title="Freight per item (top 1% clipped)", xlabel="R$", ylabel="items")
sample = df.dropna(subset=["distance_km"]).sample(5000, random_state=1)
axes[1].scatter(sample["distance_km"], sample[TARGET], s=4, alpha=0.3)
axes[1].set(title="Freight vs distance (5,000 items)", xlabel="distance (km)", ylabel="R$", ylim=(0, 100))
plt.tight_layout()
plt.show()

## Look at the dataset
This is the table the model learns from: **one row per order item**. The features are on the left, and the target `freight_value` is the last column.

In [ ]:
dataset = df[FEATURES + [TARGET]]

print(f"Dataset: {dataset.shape[0]:,} rows x {dataset.shape[1]} columns (one row per order item)")
print("Features:", ", ".join(FEATURES))
print("Target  :", TARGET)

print("\nFirst 10 rows:")
display(dataset.head(10))

print("Summary statistics of the numeric columns:")
display(dataset.describe().T.round(2))

print("Missing values per column (only columns that have some):")
missing = dataset.isna().sum()
display(missing[missing > 0].to_frame("missing values"))

## 5. Split by time
The model will price **future** orders, so we train on the past and test on later months, never at random.

In [ ]:
cutoff = pd.Timestamp("2018-06-01")
train = df[df["purchase_timestamp"] < cutoff]
test = df[df["purchase_timestamp"] >= cutoff]
X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]
print(f"train: {len(train):,} items (before {cutoff.date()})   test: {len(test):,} items")

## 6. Train and compare three models
* **Baseline**: always predict the median freight. Any useful model must beat it.
* **Linear (ridge)**: one-hot categories plus scaled numbers.
* **Gradient boosting**: hundreds of small trees, each correcting the previous ones.

Both real models learn `log(1 + freight)`, because the target is skewed. `TransformedTargetRegressor` converts predictions back to reais.

In [ ]:
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

def log_target(regressor):
    return TransformedTargetRegressor(regressor=regressor, func=np.log1p, inverse_func=np.expm1)

linear = log_target(Pipeline([
    ("preprocess", ColumnTransformer([
        ("categorical", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL),
        ("numeric", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), NUMERIC),
    ])),
    ("model", Ridge(alpha=1.0)),
]))

boosting = log_target(Pipeline([
    ("preprocess", ColumnTransformer([
        ("categorical", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), CATEGORICAL),
        ("numeric", "passthrough", NUMERIC),
    ])),
    ("model", HistGradientBoostingRegressor(
        learning_rate=0.05, max_iter=600, max_leaf_nodes=63, min_samples_leaf=40,
        categorical_features=list(range(len(CATEGORICAL))), early_stopping=True,
        validation_fraction=0.1, random_state=42)),
]))

def evaluate(y_true, y_pred):
    errors = np.abs(y_true - y_pred)
    return {"MAE (R$)": mean_absolute_error(y_true, y_pred),
            "RMSE (R$)": np.sqrt(mean_squared_error(y_true, y_pred)),
            "R2": r2_score(y_true, y_pred),
            "within R$5": (errors <= 5).mean()}

models = {"baseline": DummyRegressor(strategy="median"), "linear": linear, "boosting": boosting}
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    results[name] = evaluate(y_test.to_numpy(), model.predict(X_test))
    print(f"trained {name}")

pd.DataFrame(results).T.style.format({"MAE (R$)": "{:.2f}", "RMSE (R$)": "{:.2f}", "R2": "{:.3f}", "within R$5": "{:.1%}"})

## 7. Explain the best model
**Left:** points on the diagonal are perfect predictions. **Right:** permutation importance shuffles one feature at a time and measures how much worse the error gets.

In [ ]:
from sklearn.inspection import permutation_importance

best = models["boosting"]
predictions = best.predict(X_test)

sample = X_test.sample(3000, random_state=42)
importance = permutation_importance(best, sample, y_test.loc[sample.index],
                                    scoring="neg_mean_absolute_error", n_repeats=3, random_state=42, n_jobs=-1)
importance = pd.Series(importance.importances_mean, index=FEATURES).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
limit = float(np.percentile(y_test, 99))
axes[0].scatter(y_test, predictions, s=4, alpha=0.25)
axes[0].plot([0, limit], [0, limit], color="black", linewidth=1)
axes[0].set(xlim=(0, limit), ylim=(0, limit), xlabel="actual freight (R$)", ylabel="predicted (R$)",
            title="Predicted vs actual (test set)")
importance.plot.barh(ax=axes[1], title="Permutation importance (extra MAE in R$)")
plt.tight_layout()
plt.show()

## 8. Save the model
`joblib` saves the whole pipeline (preprocessing and model) in one file, so it can be loaded later without retraining.

In [ ]:
import joblib

joblib.dump(best, "freight_model.joblib")
print("Saved freight_model.joblib")

## 9. Try it: Gradio app
Gradio turns the `predict` function into a small web app. Pick the states (the distance is estimated
from them when you leave it at 0), describe the item, and press **Submit**. The app opens below this
cell and at http://127.0.0.1:7860.

In [ ]:
import gradio as gr

# Typical distance for shipments inside one state, learned from the data
within_state_km = df[df["same_state"] == 1].groupby("seller_state")["distance_km"].median().clip(lower=20).to_dict()
CATEGORIES = sorted(df["product_category"].unique())


def estimate_distance(seller_state, customer_state):
    """Rough distance for the app: typical within-state distance, or the distance between state centres."""
    if seller_state == customer_state:
        return float(within_state_km.get(seller_state, 50.0))
    s, c = state_centres.loc[seller_state], state_centres.loc[customer_state]
    return float(haversine_km(s["geolocation_lat"], s["geolocation_lng"], c["geolocation_lat"], c["geolocation_lng"]))


def predict_freight(seller_state, customer_state, category, price, weight_g, length_cm, height_cm, width_cm,
                    items_in_order, distance_km):
    if not distance_km:
        distance_km = estimate_distance(seller_state, customer_state)
    row = pd.DataFrame([{
        "seller_state": seller_state, "customer_state": customer_state, "product_category": category,
        "price": price, "product_weight_g": weight_g, "product_length_cm": length_cm,
        "product_height_cm": height_cm, "product_width_cm": width_cm,
        "volume_cm3": length_cm * height_cm * width_cm, "distance_km": distance_km,
        "same_state": float(seller_state == customer_state), "items_in_order": items_in_order,
        "items_from_seller": items_in_order, "purchase_month": float(pd.Timestamp.now().month),
    }])[FEATURES].astype({col: "float64" for col in NUMERIC})
    freight = float(best.predict(row)[0])
    return f"## R$ {freight:.2f}\nPredicted shipping cost, {seller_state} → {customer_state}, about {distance_km:,.0f} km."

demo = gr.Interface(
    fn=predict_freight,
    inputs=[
        gr.Dropdown(STATES, value="SP", label="Seller state"),
        gr.Dropdown(STATES, value="RJ", label="Customer state"),
        gr.Dropdown(CATEGORIES, value="furniture_decor", label="Product category"),
        gr.Number(value=249.0, label="Price (R$)"),
        gr.Number(value=9500, label="Weight (g)"),
        gr.Number(value=60, label="Length (cm)"),
        gr.Number(value=40, label="Height (cm)"),
        gr.Number(value=40, label="Width (cm)"),
        gr.Number(value=1, precision=0, label="Items in the parcel"),
        gr.Number(value=0, label="Distance (km) - 0 = estimate from the states"),
    ],
    outputs=gr.Markdown(),
    title="Olist shipping-cost predictor (local model)",
    examples=[
        ["SP", "SP", "health_beauty", 59.9, 800, 30, 10, 20, 1, 25],
        ["SP", "RJ", "furniture_decor", 249.0, 9500, 60, 40, 40, 1, 360],
        ["SP", "AM", "computers_accessories", 120.0, 1500, 35, 15, 25, 1, 2700],
    ],
)
demo.launch()